# Bölüm 19 · Doğrulayıcı Faktör Analizi

[Atölye](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b19/) · [Veri ve lisans](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b19/VERI.md) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b19/GOREVLER.md) · [Raporlama](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b19/RAPORLAMA.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b19/COZUMLER.md)

**Başlangıç:** Google hesabınızla açın, **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU yeterlidir. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk kurulum internet ve birkaç dakika gerektirebilir. Özellikle bootstrap ve yapısal modeller daha uzun sürebilir.

Bu defter, aynı GitHub deposundaki ana Python analizini ve bölümün ileri eklerini çalıştırır. Dosya yüklemeniz veya önce başka bölümü elle yürütmeniz gerekmez. Gerçek veri, öğretim alt kümesi ve simülasyon ayrımı veri rehberinde açıklanır; bunları aynı örneklem türü saymayın. R/SPSS/Amos burada çalıştırılmaz.

## Üç çekirdek kazanım ve ileri rota

1. DFA modelinin serbest ve sabit yollarını belirtir.
2. Global uyumu yükler ve artıklarla birlikte yorumlar.
3. Puan güvenirliği ile yapı/geçerlik kanıtını ayırır.

**İleri rota:** ML bilgi matrisi, CR/AVE delta aralıkları ve alternatif modeller; Bölüm 18 ve olabilirlik köprüsünden sonra.

**Ç1 · Çıkış sorusu:** CFI=.96 olması modelin kuramsal olarak doğru ve nedensel olduğunu tek başına gösterir mi?

Yanıtınızı çalışma sonunda öğrenci karar günlüğüne yazın. [Rubrik ve teslim rehberi](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b19/RUBRIK.md).

## 1 · Kaynakları hazırla

Sabit bir GitHub sürümündeki öğrenci paketi SHA256 ile doğrulanır. Her çalıştırma ayrı bir geçici çalışma kopyası açar; önceki sonuçlarınız silinmez. Kod ve veri, bu doğrulanan sürümden gelir. Oturum kapanmadan sonuçları indirin.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, urllib.request, zipfile, io, tempfile, subprocess, importlib.metadata
DATA_COMMIT = '23acf4be23823aa139523c8f4db34cff248c285d'
ARCHIVE_SHA256 = 'ce6440f5ee3b4dde9709e1fb250e7b7d8ac86be101b6ababacdab240443560f6'
ARCHIVE_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/ogrenci-paketi-22-bolum.zip"
cache = Path.cwd()/"kitap-colab-kaynak.zip"
if cache.exists():
    raw = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError("Kaynak indirilemedi. İnterneti kontrol edip hücreyi yeniden çalıştırın: "+ARCHIVE_URL) from exc
if hashlib.sha256(raw).hexdigest()!=ARCHIVE_SHA256:
    raise ValueError("Öğrenci paketi SHA256 denetimi başarısız. Kontrolü kaldırmayın; yerel kitap-colab-kaynak.zip dosyasını doğrulayın.")
cache.write_bytes(raw)
work = Path(tempfile.mkdtemp(prefix="b19-atolye-",dir=Path.cwd()))
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(z.infolist())<10000 and sum(i.file_size for i in z.infolist())<134217728
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if work.resolve() not in target.parents or ((member.external_attr>>16)&0o170000)==0o120000:
            raise ValueError("Güvenli olmayan arşiv yolu")
    z.extractall(work)
root = work/"ileri-istatistik-22-bolum"
assert (root/"BOLUM_ESLESMESI.json").is_file()
application=root/'uygulamalar/b19-dogrulayici-faktor'
modules=['uygulamalar/b19-dogrulayici-faktor/puan-guvenirligi']
expected_main=json.loads((root/"bolumler/b19/beklenen.json").read_text())
expected_extra={m:json.loads((root/m/"sonuclar/ozet.json").read_text()) for m in modules}
print("Kaynak doğrulandı; Bölüm 19 çalışma kopyası hazır.")

## 2 · Analiz ortamını hazırla

Sınanmış paket sürümleri kullanılır. Mevcut çekirdek bu sürümlerdeyse doğrudan kullanılır. Farklıysa analizler ayrı bir Python ortamında kurulur; Colab çekirdeğindeki paketler topluca değiştirilmez. Kurulum başarısız olursa hücredeki hatayı inceleyin. Python 3.12 veya üzeri çalışma zamanı önerilir. GPU gerekmez.

In [ ]:
FORCE_ISOLATED=globals().get("FORCE_ISOLATED",False)
PINS={'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0', 'matplotlib': '3.11.2', 'statsmodels': '0.15.0', 'PyMuPDF': '1.26.6'}
def package_versions(executable):
    check="import importlib.metadata,json; names="+repr(list(PINS))+";print(json.dumps({n:importlib.metadata.version(n) for n in names}))"
    attempt=subprocess.run([str(executable),"-c",check],capture_output=True,text=True)
    return json.loads(attempt.stdout) if attempt.returncode==0 else {}
python=Path(sys.executable)
if FORCE_ISOLATED or package_versions(python)!=PINS:
    environment=Path.cwd()/".kitap-colab-python"
    if not (environment/"bin/python").exists():
        subprocess.run([sys.executable,"-m","venv",str(environment)],check=True)
    python=environment/"bin/python"
    if package_versions(python)!=PINS:
        subprocess.run([str(python),"-m","pip","install",*[name+"=="+version for name,version in PINS.items()]],check=True)
assert package_versions(python)==PINS,"Paket sürümleri sınanmış ortamla eşleşmiyor"
versions=package_versions(python)
run_env={**os.environ,"MPLBACKEND":"Agg","MPLCONFIGDIR":str(work/"mpl"),"OPENBLAS_NUM_THREADS":"1","OMP_NUM_THREADS":"1","PYTHONDONTWRITEBYTECODE":"1"}
(work/"logs").mkdir(exist_ok=True)
def run_script(relative):
    print("Çalışıyor:",relative,flush=True)
    result=subprocess.run([str(python),str(root/relative)],cwd=root,env=run_env,capture_output=True,text=True,timeout=900)
    (work/"logs"/(relative.replace("/","__")+".log")).write_text(result.stdout+"\n"+result.stderr)
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+"\n"+result.stderr[-4000:])
    print("Tamamlandı.")
print("Analiz ortamı hazır:",versions)

## 3 · Ortak faktör analizi girdilerini üret

Bu bölüm Bölüm 18'in sabit A/B bölmesini ve ortak veri dosyalarını kullanır. Aşağıdaki adım aynı kaynaktan bunları yeniden üretir; başka bir defter açmanız gerekmez. Bu işlem yeni örneklem veya dış doğrulama oluşturmaz.

In [ ]:
run_script("bolumler/b18/analiz.py")

## Ana analiz

Kaynak ve yöntem için yukarıdaki veri rehberini okuyun. Ana analiz ve sonradan eklenen duyarlılık uygulamaları ayrı çalıştırılır. Sonuçlar önceden gösterilen tabloların kopyası olarak değil, kod yeniden yürütülerek üretilir.

In [ ]:
before_figures=set((root/"assets_figures").glob("*.pdf"))
run_script("bolumler/b19/analiz.py")
main_result=json.loads((application/"sonuclar/ozet.json").read_text())
from IPython.display import display, JSON, HTML, Image
print("Ana sonuçlar — ayrıntıları açmak için okları kullanın:")
display(JSON(main_result,expanded=False))

## İleri uygulamalar ve duyarlılık

Her ekin açıklaması atölye sayfasındadır. Bu sonuçları ana analizle veya birbirinden farklı tahmin hedefleriyle karıştırmayın. Hazır kontrol betikleri varsa önce onlar, ardından ek analiz çalışır.

In [ ]:
extra_results={}
for module in modules:
    if (root/module/"dogrula.py").is_file(): run_script(module+"/dogrula.py")
    run_script(module+"/analiz.py")
    extra_results[module]=json.loads((root/module/"sonuclar/ozet.json").read_text())
    print(module)
    display(JSON(extra_results[module],expanded=False))

## Sonuçları referanslarla karşılaştır

Kitabın sayısal referansları, önceki sürümden değişmeden alınır. Sürüm/zaman gibi hesap dışı alanlar mevcut karşılaştırma politikasına göre dışlanır. Hata oluşursa veri, sürüm ve günlük dosyalarını kontrol edin; referansları sonuca uydurmayın. Bu kontrol bilimsel varsayımları veya SPSS yürütmesini doğrulamaz.

In [ ]:
sys.path.insert(0,str(root/"tools"))
from dogrula_22 import compare
checks={"main":compare(expected_main,main_result)}
for module in modules:
    checks[module]=compare(expected_extra[module],json.loads((root/module/"sonuclar/ozet.json").read_text()))
assert all(not differences for differences in checks.values()),checks
print("Ana analiz ve",len(modules),"ileri ek için sayısal referanslar eşleşti.")

## Grafikler ve özet tablolar

Grafikler bu çalıştırmada üretilir. Ön koşul olarak çalıştırılan Bölüm 18'in şekilleri burada tekrar gösterilmez. CSV tablolarının ilk 12 satırı görüntülenir; tam dosyalar çıktı paketindedir. Şekil sayısı bölüme göre değişir.

In [ ]:
import csv,html
new_figures=sorted(set((root/"assets_figures").glob("*.pdf"))-before_figures)
for figure in new_figures:
    png=figure.with_suffix(".png")
    renderer="import fitz,sys;d=fitz.open(sys.argv[1]);d[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(sys.argv[2])"
    subprocess.run([str(python),"-c",renderer,str(figure),str(png)],check=True)
    print(figure.name)
    display(Image(filename=str(png)))
for table in sorted((application/"sonuclar").glob("*.csv"))[:3]:
    with table.open(encoding="utf-8-sig",newline="") as f: rows=list(csv.reader(f))
    content="<h4>"+html.escape(table.name)+"</h4><table>"
    for i,row in enumerate(rows[:13]):
        tag="th" if i==0 else "td"
        content+="<tr>"+"".join("<"+tag+">"+html.escape(value)+"</"+tag+">" for value in row)+"</tr>"
    display(HTML(content+"</table>"))

## Sıra sizde

# Bölüm 19 — görevler

Güncel Revizyon 22 görevleri; önce kendiniz çözün.

## D1–D3: Modelden Önce

1.  Bu bölümde yarım A’yı kullanmamanın ve modelleri yarım B açılmadan yazmanın yararı nedir? Buna rağmen kalan doğrulama sınırı nedir?

2.  Ters kodlanan maddeleri ve kişi sayısını belirtin. S matrisinin köşegeninin 1 olması faktör varyanslarının kendiliğinden 1 olduğu anlamına gelir mi?

3.  ML, sağlam ML, WLSMV ve FIML kavramlarından hangileri bu uygulamada gerçekten kullanılmıştır?

Yanıtlar Ek F, Bölüm 19’dadır (s. kitaptaki ilgili sayfa).

# Rehberli ve Bağımsız Etkinlikler

## G1–G4: Rehberli Çözüm

1.  M1’in 55 momentini, 21 parametresini ve 34 serbestlik derecesini açıklayın. M2’nin sd’si neden 31’dir? Varyans-1 ve işaretleyici-yük ölçeklendirmelerini ayırın.

2.  M1 için F’den T’yi; başlangıç modelinden CFI’yi ve n paydasıyla RMSEA’yı yeniden hesaplayın. RMSEA aralığının alt sınırı ne söyler?

3.  A1’in standart yükünden $R^2$ ve artık varyansını bulun. Yük aralığının sıfırı dışlaması neden güçlü madde demek değildir?

4.  A alt ölçeğinin M1’deki AVE ve CR’sini yük tablosundan hesaplayın. İki payın farkını ve model varsayımlarını açıklayın.

## B1–B3: Bağımsız Çözüm

1.  M0–M1 farkını ve serbestlik derecesini hesaplayın; M0’ı CFI başlangıç modelinden ayırın.

2.  M1–M2 farkını ve M2 uyum özetini bulun. Bu karşılaştırmayı, M1’in MI listesine bakıp en büyük üç değeri eklemekten ayıran nedir?

3.  M2’de C’nin CR’si neden M1’dekinden düşüktür? $\hat\phi^2$ ile AVE karşılaştırmasından Türkçe ölçeğin geçerli olduğu çıkar mı?

## H1: Hatalı Raporu Onarma

Beş iddiayı sonuç ve yapılmayan işlem ayrımıyla düzeltin.

## P1: Bağımsız Doğrulama Protokolü

Yeni bir örneklemde (örneğin Türkçe formun uygulandığı bir grupta) sınanacak model, önceden gerekçelendirilmiş alternatifler, hedef grup, örneklem planı, eksik veri/kestirici kararları ve modifikasyon sınırlarını içeren bir sayfalık protokol yazın. A1 ve A4 içeriği ile bu bölümde öngörülmeden görülen A4–C artıkları için kararları sonuçları görmeden tanımlayın. Gerçekte yapılmamış veri toplama veya uzman görüşmesini yapılmış gibi yazmayın. Rubrik: kaynak/ön işlem 20, model/hesap 30, belirsizlik/yorum 25, bağımsız doğrulama planı 15, yeniden üretim 10 puan.

## T1–T3: Puan ve Aralık Denetimi

1.  M1 A için ham toplam payı .706272 ile standart toplam payı .716047 neden farklıdır? Model veri-standart birimde kurulmuşken ham toplamın ağırlıkları nasıl yazılır?

2.  CR/AVE aralığında neden tam parametre kovaryansı gerekir? M1 A’nın CR aralığı $[.692920,.739173]$ iken yalnız .70 eşiğine bakarak onay vermenin ve M1/M2 aralık örtüşmesini fark testi saymanın sorununu açıklayın.

3.  M1’in iki AVE–$\phi^2$ farkı için Bonferroni aralıklarını yorumlayın. Ortak kapsama hangi aile içindir; hangi geçerlik ve doğrulama iddiaları yapılamaz?

P1 dosyasına puan ağırlıklarını, hedef faktör payının hata tanımını, tam kovaryansla belirsizliği ve analizlerin ne zaman geliştirildiğini ekleyin.

# Alıştırmalar

## Kolay Düzey

1.  DFA ile AFA arasındaki temel fark nedir?

2.  CFI ve TLI neyi gösterir?

3.  RMSEA arttıkça ne düşünülür?

4.  AVE ne için kullanılır?

5.  Modifikasyon indeksleri nasıl kullanılmalıdır?

## Orta Düzey

1.  DFA modelinde CFI=.88, TLI=.86, RMSEA=.091 bulunmuştur. Açıklayınız.

2.  Bir maddenin standartlaştırılmış faktör yükü 0.39’dur. Hemen çıkarılmalı mıdır?

3.  AMOS iki hata terimi arasında kovaryans önermektedir. Hangi durumda savunulabilir?

4.  Uyum iyi fakat bir faktörde AVE 0.50’nin altındadır. Ne düşünülmelidir?

5.  DFA sonuçları neden tek geçerlik kanıtı gibi sunulmamalıdır?

## İleri Düzey

1.  Modifikasyon indeksleriyle model sürekli iyileştiriliyorsa risk nedir?

2.  Faktör korelasyonu $r=.92$ ise hangi geçerlik sorunu akla gelir?

3.  Çok çarpık ordinal Likert maddeleriyle DFA tahmin edilmiştir. Hangi konu düşünülmelidir?

4.  Kuramsal olarak güçlü bir maddenin yükü orta, hata varyansı yüksektir. Karar nasıl verilmelidir?

5.  Güçlü bir DFA raporunda en az beş unsur yazınız.

Yöntem belgelerine erişim: 6 Eylül 2026. Veri kaynağı ve bölme kaydı Bölüm <a href="#chap:faktor" data-reference-type="ref" data-reference="chap:faktor">[chap:faktor]</a> uygulama klasöründe belgelenmiştir.


Önce kendi yorumunuzu yazın; sonra [çözümlerle](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b19/COZUMLER.md) karşılaştırın. Sonuç, belirsizlik, varsayım ve sınırlılıkları [raporlama rehberine](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b19/RAPORLAMA.md) göre ayırın.

In [ ]:
yanitlarim={"bulgular":"", "belirsizlik":"", "varsayimlar":"", "sinirliliklar":""}
# Metinleri doldurun; sayısal kontrolü geçmek bu yorumları tamamlamaz.

## Çıktıları indir

Son hücre sonuç JSON/CSV'lerini, grafiklerin PDF/PNG dosyalarını, günlükleri ve kendi yanıtlarınızı paketler. Colab'ın solundaki Dosyalar panelini yenileyin ve aşağıda yazan ZIP'i indirin. Geçici çalışma zamanı kapanmadan bunu yapın; kod Drive'a otomatik erişmez.

## Öğrenci kararı ve kısa savunma

**Karar görevi:** M1 veya önceden yazılmış M2 modelini seçin; uyum ve CR değişimini model varsayımıyla birlikte raporlayın.

Yalnız bir kararı değiştirin. Ana sonucu koruyun; gerekirse yeni bir hesap hücresi ekleyip denemeyi ayrı kaydedin. Sayısal referansları yeni sonuca uydurmayın. Önce/sonra kanıtını sonuç dosyası, tablo veya açık bir hesapla ilişkilendirin. Yapılmamış hesabı yapılmış gibi yazmayın.

**3–5 dakikalık savunma:** Hangi hedefi seçtiniz ve neden? Tam olarak ne değişti? Hangi belirsizlik ve kapsam sınırı kaldı? Karşılaştırma aynı hipotezi mi sınadı?

Aşağıdaki alanları kendi çalışmanızla doldurun. **Tümünü çalıştır**, bu alanları sizin yerinize tamamlamaz. Sayısal kontrollerin geçmesi öğrenci kararının tamamlandığı anlamına gelmez. Alanların dolu olması da kanıtın doğruluğunu otomatik onaylamaz.

In [ ]:
karar_gunlugu = {
    "bolum": 19,
    "gorev": 'M1 veya önceden yazılmış M2 modelini seçin; uyum ve CR değişimini model varsayımıyla birlikte raporlayın.',
    "kararim": "",
    "gerekcem": "",
    "degistirdigim_tek_sey": "",
    "onceki_kanit": "",
    "sonraki_kanit": "",
    "belirsizlik_ve_sinir": "",
    "kisa_savunma": "",
    "cikis_yanitim": ""
}
zorunlu_alanlar = ["kararim","gerekcem","degistirdigim_tek_sey","onceki_kanit","sonraki_kanit","belirsizlik_ve_sinir","kisa_savunma","cikis_yanitim"]
karar_gunlugu["alanlar_dolu"] = all(isinstance(karar_gunlugu[k],str) and bool(karar_gunlugu[k].strip()) for k in zorunlu_alanlar)
karar_gunlugu["durum"] = "Öğretim elemanı incelemesi gerekir" if karar_gunlugu["alanlar_dolu"] else "Öğrenci kararı henüz tamamlanmadı"
print(karar_gunlugu["durum"])
# Bu hücreyi doldurduktan sonra çıktı ZIP'i hücresini yeniden çalıştırın.

In [ ]:
manifest={"chapter":19,"source_commit":DATA_COMMIT,"source_archive_sha256":ARCHIVE_SHA256,"versions":versions,"checks":checks,"answers":yanitlarim,"scope":"Python ana analiz ve ileri ekler; SPSS/R/Amos çalıştırılmadı"}
# OGRETIM_EXPORT_BEGIN
manifest["ogrenci_karari"] = karar_gunlugu
(work/"ogrenci-karari.json").write_text(json.dumps(karar_gunlugu,ensure_ascii=False,indent=2),encoding="utf-8")
# OGRETIM_EXPORT_END
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive=Path.cwd()/"b19-colab-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(work/"ogrenci-karari.json","ogrenci-karari.json")
    z.write(work/"calisma-kaydi.json","calisma-kaydi.json")
    for module in ['uygulamalar/b19-dogrulayici-faktor',*modules]:
        for f in sorted((root/module/"sonuclar").rglob("*")):
            if f.is_file() and f.suffix in [".json",".csv"]:z.write(f,str(f.relative_to(root)))
    for figure in new_figures:
        for f in [figure,figure.with_suffix(".png")]: z.write(f,"grafikler/"+f.name)
    for f in (work/"logs").glob("*.log"):z.write(f,"logs/"+f.name)
print("İndirmeye hazır:",archive.name)